# 07 — Direcção da adjudicação

Quando o autor e a maioria do painel divergem, em que sentido? A repartição confirmação/substituição por padrão, e a verificação de integridade da semente congelada.

In [1]:
%run 00-setup.ipynb   # funções partilhadas e configuração da suite

setup pronto: run 2 | zip: analysis-export.zip


In [2]:
RUN_ID = REFERENCE_RUN   # = 2, a execucao de referencia (a adjudicada); mude para analisar outra
ann    = load("annotations.csv", run=RUN_ID)
gold   = load("gold.csv", run=RUN_ID)
status = load("status.csv", run=RUN_ID)
fail   = load("failures.csv", run=RUN_ID)
dist   = load("pattern_distribution.csv", run=RUN_ID)
sample = load("sample.csv", run=RUN_ID)
meta   = load("meta.csv", run=RUN_ID).iloc[0].to_dict()

print(f"run {RUN_ID}: {meta['label']}  |  painel: {meta['panel']}")
print(f"{len(ann)} linhas de anotacao, {len(gold)} linhas de gold, {len(status)} linhas de estado, {len(fail)} linhas de falha")

run 2: llm_panel #2  |  painel: deepseek/deepseek-v4-flash;meta-llama/llama-3.3-70b-instruct;openai/gpt-oss-120b;qwen/qwen3-next-80b-a3b-instruct
67317 linhas de anotacao, 5000 linhas de gold, 3600 linhas de estado, 57 linhas de falha


## Que passagem de adjudicação? (`PASS`)
A referência conserva **duas passagens**. `open` é o *consenso adjudicado*: o autor adjudicou com a maioria do painel pré-preenchida no ecrã (eficiente, executável em escala, mas a seguir o painel). `blind` é o autor a adjudicar apenas a partir do texto da revisão (independente do painel, mas de menor dimensão). A análise de validade B e a análise de direcção C, abaixo, respeitam ambas este selector, pelo que se pode re-executar o notebook inteiro contra qualquer das referências. Veja o [notebook da passagem cega](/notebooks/anchoring.ipynb) para aferir quanto as duas divergem.

In [3]:
PASS = "open"   # "open" (painel no ecra, em escala) ou "blind" (independente, menor)

## Análise C: direcção de adjudicação (derivada por *run*)
Por padrão, com que frequência o autor **confirmou** a maioria do painel desta *run* face a quantas vezes a **substituiu**. *Derivamo-la*, comparando `final_label` com a maioria **desta *run***, em vez de ler a coluna `direction` armazenada, porque a coluna armazenada está vinculada à *run* que esteve no ecrã no momento da adjudicação. Uma *adição independente* é uma célula que a maioria do painel deu como ausente mas o autor deu como presente.

In [4]:
adjudication_direction(ann, gold, PASS)

adicoes independentes (painel ausente, autor presente): 234 / 4335 celulas


dir,confirmation,replacement,override_rate
code,,,
PE-2,192,37,0.162
TM-1,191,37,0.162
PM-1,201,27,0.118
PE-3,205,23,0.101
PE-1,210,18,0.079
SE-3,211,17,0.075
DR-2,211,17,0.075
PM-2,215,13,0.057
PM-4,216,12,0.053


> Uma confirmação **não** é «o painel estava certo», mas sim «o painel e o autor concordaram». A referência é o juízo do autor; o painel é o instrumento que está a ser medido. Uma `override_rate` elevada num código indica que o painel lê mal, de forma sistemática, essa família.

## Verificação de integridade: será fiel a maioria derivada?
Derivamos a maioria a partir das anotações *vivas*, e não da `panel_seed_at_adjudication` armazenada. Como a execução de referência é a que foi **adjudicada**, as duas têm de coincidir sobre ela, pois a semente *foi* congelada a partir das suas anotações. O `seed.csv` conserva os votos congelados de cada modelo, e, como a execução de referência é a adjudicada, a verificação incide directamente sobre ela.

In [5]:
seed = load("seed.csv")   # a semente congelada no momento da adjudicacao (run de referencia)
chk = seed.merge(ann[["individualId","code","modelSlug","present"]],
                 on=["individualId","code","modelSlug"], suffixes=("_seed","_live"), how="inner")
agree = (chk["present_seed"] == chk["present_live"]).mean() if len(chk) else float("nan")
print(f"semente == anotacoes vivas em {agree:.1%} de {len(chk)} celulas correspondidas")

semente == anotacoes vivas em 100.0% de 513 celulas correspondidas


> **100%**: derivar é fiel ao que o autor efectivamente viu, pelo que a semente armazenada nunca tem de ser lida. Numa *run* que *não* foi adjudicada, a coincidência é menor (o seu painel difere do que esteve no ecrã), e a direcção dessa *run* é um *contrafactual*, o que é aceitável, desde que assim assinalado.